## Detrending

This notebook updates the `Lumen detrending.ipynb`.

**Main changes:** Updating the temperature and RH/dew-point sections to use `*__(new_core).nc` files instead. Otherwise, data flow is untouched.?

# 0. Initial setup

### Import libraries
Import all libraries needed to complete the calculations in this notebook and plot results.

In [ ]:
import os
os.chdir('..')

import xarray as xr
import pandas as pd
import numpy as np
import panel as pn
pn.extension()
import hvplot.xarray
import hvplot.pandas
import holoviews as hv
from bokeh.models import HoverTool
import datetime


### Common inputs
Select common inputs and input files needed to process data.

In [2]:
date_stamp = datetime.date.today().strftime('%Y%m%d')

station_list = pd.read_csv("inputs/weather_stations.csv")
station_list = station_list[station_list['STATION'] == 'SACRAMENTO'].reset_index(drop=True)

station_weights = pd.read_csv("inputs/station_weights_04122019.csv")
station_weights = station_weights[['PA_NAME','STATION','MONTH','PA_WEIGHT']]

simulation_list = pd.read_csv("inputs/simulation_list_initial_wrf.csv") 

variants = pd.read_csv("inputs/variantsID_mapping_initial_wrf.csv", index_col=['scenario','simulation','variantYear'])

# 1. Hourly de-trended temperature library

### Step 1a. Load hourly temperature projections
Re-load localized WRF data previously pulled from Cal-Adapt Analytics Engine platform.  

In [3]:
data_aggregated = None

simulation_rename = {
    'wrf_ucla_mpi-esm1-2-hr_ssp370_r3i1p1f1': 'WRF_MPI-ESM1-2-HR_r3i1p1f1',
    'wrf_ucla_miroc6_ssp370_r1i1p1f1': 'WRF_MIROC6_r1i1p1f1',
    'wrf_ucla_ec-earth3-veg_ssp370_r1i1p1f1': 'WRF_EC-Earth3-Veg_r1i1p1f1',
    'wrf_ucla_ec-earth3_ssp370_r1i1p1f1': 'WRF_EC-Earth3_r1i1p1f1',
    'wrf_ucla_taiesm1_ssp370_r1i1p1f1': 'WRF_TaiESM1_r1i1p1f1',
}

for station_name in station_list['station_name'].values:
    print(station_name)
    ds = xr.open_dataset(f'inputs/{station_name}_temp_(new_core).nc')
    ds = ds.rename({'sim': 'simulation'})
    ds = ds.sel(simulation=list(simulation_rename.keys()))
    ds = ds.assign_coords(simulation=[simulation_rename[s] for s in ds['simulation'].values])
    data = ((ds[station_name] - 273.15) * 9/5 + 32).expand_dims(scenario=['Historical + SSP 3-7.0 -- Business as Usual']).to_dataset(name=station_name)
    data_aggregated = data if data_aggregated is None else xr.merge([data_aggregated, data])

data_aggregated = data_aggregated.to_array(dim='station').to_dataset(name='degF')
data_aggregated = data_aggregated.transpose('scenario', 'simulation', 'station', 'time')
#data_aggregated['degF'].hvplot.line(x='time', xlabel='', ylabel='degF', grid=True, ylim=(-10,140), yticks=15, xticks = 10, color='blue', alpha=0.5)

# Filter for simulations flagged to include
simulation_to_keep = simulation_list[simulation_list['include'] == 1]['simulation'].to_list()
simulation_to_keep = [s for s in simulation_to_keep if s in data_aggregated['simulation'].values]
data_aggregated = data_aggregated.sel(simulation=simulation_to_keep)

del data
del ds


Sacramento Executive Airport (KSAC)


### Step 1b. Assign quantile bins
Calculate hourly rank order in each year and assign quantile bins

In [4]:
data_aggregated['rank'] = data_aggregated['degF'].groupby('time.year').apply(lambda x: x.argsort().argsort() + 1)
data_aggregated['bin'] = xr.DataArray("",coords=data_aggregated['degF'].coords, dims=data_aggregated['degF'].dims)
data_aggregated['year'] = data_aggregated['time.year']

quantile_bins = pd.read_csv("inputs/quantile_bins.csv", index_col='bin')

for bin in quantile_bins.index:
    mask = (data_aggregated['rank'] >= quantile_bins.loc[bin,'rank_start']) & (data_aggregated['rank'] <= quantile_bins.loc[bin,'rank_end']) 
    data_aggregated['bin'] = xr.where(mask, bin, data_aggregated['bin'])
    
quantiles = data_aggregated.to_dataframe().groupby(['scenario','simulation','station','year', 'bin'])['degF'].mean()
quantiles = quantiles.to_xarray().to_dataset()

### Step 1c. Calculate trends by quantile
Calculate trends for each quantile based on 2nd-order polynomial fit over the 1981-2100 period 

In [5]:
quantile = quantiles.sel(year=slice(1981,2100))
quantileFit = xr.polyval(quantile['year'], quantile.polyfit('year', 2).degF_polyfit_coefficients).to_dataset(name='degF')
quantile.to_dataframe().pivot_table(columns=['scenario','simulation','station'], index=['year','bin']).to_csv('outputs/quantiles_1981-2100.csv')
quantileFit.to_dataframe().pivot_table(columns=['scenario','simulation','station'], index=['year','bin']).to_csv('outputs/quantileFit_2ndorder_polynomial.csv')

### Step 1d. Create de-trended data library
For base year (2022) through 2050, calculate de-trended hourly temperatures for each variant using +/-25 years around forecast years. 

In [6]:
hourlytemp_df = data_aggregated.to_dataframe()
# del data_aggregated
hourlytemp_df = hourlytemp_df.reorder_levels(['scenario','simulation','station','time'])
hourlytemp_df.set_index(['year', 'bin'], append=True, inplace=True)
hourlytemp_df['degF'] = hourlytemp_df['degF'].astype('float32')

In [7]:
hourlytemp_detrended_allyears = None
delta = 25

for year in range(2022, 2051):
    print(year)
    quantileFit_y = quantileFit.sel(year=year)
    quantile_trend = quantile.polyfit('year', 2).sel(degree=1).rename({'degF_polyfit_coefficients':'degF/yr'}) \
            + (2 * year * quantile.polyfit('year', 2).sel(degree=2).rename({'degF_polyfit_coefficients':'degF/yr'}))
    quantile_detrended = quantile - (quantileFit - quantileFit_y)

    degF_trend = quantileFit.sel(year=slice(year-delta,year+delta)) - quantileFit_y
    degF_trend = degF_trend.to_dataframe().reorder_levels(['scenario','simulation','station','year','bin']).rename(columns={'degF':'degF_trend'})
    hourlytemp_detrended = hourlytemp_df.merge(degF_trend, left_on=['scenario','simulation','station','year','bin'], right_index=True, how='inner')
    hourlytemp_detrended['degF_detrended'] = hourlytemp_detrended['degF'] - hourlytemp_detrended['degF_trend']
    hourlytemp_detrended['degF_detrended'] = hourlytemp_detrended['degF_detrended'].astype('float32')
    hourlytemp_detrended = hourlytemp_detrended.reset_index()
    hourlytemp_detrended['variantYear'] = hourlytemp_detrended['year'] - year
    hourlytemp_detrended['forecastDateTime_PST'] = pd.to_datetime(dict({'year':year,
                                                                        'month':hourlytemp_detrended['time'].dt.month, 
                                                                        'day':hourlytemp_detrended['time'].dt.day,
                                                                        'hour':hourlytemp_detrended['time'].dt.hour,
                                                                       }))
    hourlytemp_detrended['forecastDate_PST'] = hourlytemp_detrended['forecastDateTime_PST'].dt.date
    hourlytemp_detrended['Hour'] = hourlytemp_detrended['forecastDateTime_PST'].dt.hour + 1
    hourlytemp_detrended = hourlytemp_detrended.merge(variants, left_on=['scenario','simulation','variantYear'], right_index=True, how='inner')
    hourlytemp_detrended = hourlytemp_detrended.merge(station_list, left_on='station', right_on='station_name', how='inner')
    hourlytemp_detrended = hourlytemp_detrended[['STATION','forecastDateTime_PST','forecastDate_PST','Hour','variantID','degF_detrended']]
    hourlytemp_detrended = hourlytemp_detrended.set_index(['STATION','forecastDateTime_PST','forecastDate_PST','Hour','variantID'])
    hourlytemp_detrended = hourlytemp_detrended.sort_values(by=['STATION','forecastDateTime_PST','variantID'])
    hourlytemp_detrended = hourlytemp_detrended.reorder_levels(['forecastDateTime_PST','forecastDate_PST','Hour','STATION','variantID'])
    hourlytemp_detrended = hourlytemp_detrended.unstack().unstack()
    hourlytemp_detrended_allyears = pd.concat([hourlytemp_detrended_allyears, hourlytemp_detrended])
    del hourlytemp_detrended

hourlytemp_detrended_allyears

2022
2023
2024
2025
2026
2027
2028
2029
2030
2031
2032
2033
2034
2035
2036
2037
2038
2039
2040
2041
2042
2043
2044
2045
2046
2047
2048
2049
2050


degF_detrended             \
variantID                                             103        104   
STATION                                        SACRAMENTO SACRAMENTO   
forecastDateTime_PST forecastDate_PST Hour                             
2022-01-01 00:00:00  2022-01-01       1         63.046093  54.101013   
2022-01-01 01:00:00  2022-01-01       2         62.480350  52.234718   
2022-01-01 02:00:00  2022-01-01       3         58.890091  50.577988   
2022-01-01 03:00:00  2022-01-01       4         57.593552  49.694855   
2022-01-01 04:00:00  2022-01-01       5         57.058819  49.395973   
...                                                   ...        ...   
2050-12-31 19:00:00  2050-12-31       20        51.567886  50.873646   
2050-12-31 20:00:00  2050-12-31       21        53.138973  54.176403   
2050-12-31 21:00:00  2050-12-31       22        54.219799  56.509033   
2050-12-31 22:00:00  2050-12-31       23        54.983677  57.435440   
2050-12-31 23:00:00  2050-12-31       24        55.023228  57.123539   

                                                                             \
variantID                                         105        106        107   
STATION                                    SACRAMENTO SACRAMENTO SACRAMENTO   
forecastDateTime_PST forecastDate_PST Hour                                    
2022-01-01 00:00:00  2022-01-01       1     54.795013  53.602131  54.485294   
2022-01-01 01:00:00  2022-01-01       2     51.331852  49.918209  53.412304   
2022-01-01 02:00:00  2022-01-01       3     49.268013  48.040051  52.643986   
2022-01-01 03:00:00  2022-01-01       4     47.621834  45.844810  52.090111   
2022-01-01 04:00:00  2022-01-01       5     45.765625  43.954445  51.812222   
...                                               ...        ...        ...   
2050-12-31 19:00:00  2050-12-31       20    52.586113  54.829517  51.006779   
2050-12-31 20:00:00  2050-12-31       21    56.659744  57.466755  52.502720   
2050-12-31 21:00:00  2050-12-31       22    58.262199  59.328743  52.248726   
2050-12-31 22:00:00  2050-12-31       23    58.424248  59.837025  52.421432   
2050-12-31 23:00:00  2050-12-31       24    57.451584  59.816589  52.174625   

                                                                             \
variantID                                         108        109        110   
STATION                                    SACRAMENTO SACRAMENTO SACRAMENTO   
forecastDateTime_PST forecastDate_PST Hour                                    
2022-01-01 00:00:00  2022-01-01       1     54.328354  59.334366  49.160961   
2022-01-01 01:00:00  2022-01-01       2     50.655632  57.033070  49.351681   
2022-01-01 02:00:00  2022-01-01       3     48.623611  56.668762  48.998528   
2022-01-01 03:00:00  2022-01-01       4     46.784203  55.216709  49.185844   
2022-01-01 04:00:00  2022-01-01       5     45.263653  55.075153  48.919205   
...                                               ...        ...        ...   
2050-12-31 19:00:00  2050-12-31       20    44.607346  56.159542  52.315369   
2050-12-31 20:00:00  2050-12-31       21    48.388763  56.611660  55.519073   
2050-12-31 21:00:00  2050-12-31       22    50.765751  56.836605  58.058914   
2050-12-31 22:00:00  2050-12-31       23    52.300926  56.598476  55.753414   
2050-12-31 23:00:00  2050-12-31       24    52.314548  56.694443  53.154285   

                                                                  ...  \
variantID                                         111        112  ...   
STATION                                    SACRAMENTO SACRAMENTO  ...   
forecastDateTime_PST forecastDate_PST Hour                        ...   
2022-01-01 00:00:00  2022-01-01       1     56.868343  57.529510  ...   
2022-01-01 01:00:00  2022-01-01       2     53.748032  53.867149  ...   
2022-01-01 02:00:00  2022-01-01       3     53.305393  52.703377  ...   
2022-01-01 03:00:00  2022-01-01       4     52.548557  50.537193  ...   
2022-0

# 2. Temperature inputs for demand forecasting

### Step 2a. Hourly temperatures by station
For each station, create a csv file with hourly de-trended temperatures across all variants. Each file has ~52 million observations (29 years x 8,760 hours x 204 variants). Data is saved in wide format to reduce file size.

In [8]:
df = hourlytemp_detrended_allyears.stack('STATION').droplevel(level=0, axis=1)
df = df.reorder_levels(['STATION','forecastDateTime_PST','forecastDate_PST','Hour'])
df = df.sort_values(by=['STATION','forecastDateTime_PST'])
df.columns = [f'TEMP_variantID_{col}' for col in df.columns]
#df.to_csv('outputs/hourly/WS_hourly_TEMP_detrended_all_' + date_stamp + '.csv')
#df.to_hdf('outputs/hourly/WS_hourly_TEMP_detrended_all_' + date_stamp + '.h5', key='data', mode='w')

for station_name, STATION in zip(station_list['station_name'].values, station_list['STATION'].values):
    print(station_name)
    df_station = df[df.index.get_level_values('STATION') == STATION]
    df_station.to_csv('outputs/hourly/temperature/WS_hourly_TEMP_detrended_' + STATION + '_' + date_stamp + '.csv')

# Delete unused data from memory 
# del df
del df_station

/var/folders/8t/v7mdtpss51qbx7bmcywwnw740000gn/T/ipykernel_38478/1020629326.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df = hourlytemp_detrended_allyears.stack('STATION').droplevel(level=0, axis=1)


Sacramento Executive Airport (KSAC)


### Step 2b. Daily temperature stats by station
For each station, create a csv file with daily temperature statistics across all variants.

In [9]:
hourlytemp_daily_input = hourlytemp_detrended_allyears.droplevel(['forecastDate_PST','Hour'], axis = 0)

# Calculate daily temperature stats
dailymaxtemp_detrended = hourlytemp_daily_input.resample('D').max()
dailymintemp_detrended = hourlytemp_daily_input.resample('D').min()
dailymeantemp_detrended = hourlytemp_daily_input.resample('D').mean()

# Drop leap days
dailymaxtemp_detrended = dailymaxtemp_detrended[~((dailymaxtemp_detrended.index.month == 2) & (dailymaxtemp_detrended.index.day == 29))]
dailymintemp_detrended = dailymintemp_detrended[~((dailymintemp_detrended.index.month == 2) & (dailymintemp_detrended.index.day == 29))]
dailymeantemp_detrended = dailymeantemp_detrended[~((dailymeantemp_detrended.index.month == 2) & (dailymeantemp_detrended.index.day == 29))]

# Rename time axis
dailymaxtemp_detrended = dailymaxtemp_detrended.rename_axis(index='forecastDate_PST')
dailymintemp_detrended = dailymintemp_detrended.rename_axis(index='forecastDate_PST')
dailymeantemp_detrended = dailymeantemp_detrended.rename_axis(index='forecastDate_PST')

# Re-format data
dailymaxtemp_detrended = dailymaxtemp_detrended.stack().stack().reorder_levels(['STATION','forecastDate_PST','variantID']).sort_index(level=['STATION','variantID','forecastDate_PST'])
dailymintemp_detrended = dailymintemp_detrended.stack().stack().reorder_levels(['STATION','forecastDate_PST','variantID']).sort_index(level=['STATION','variantID','forecastDate_PST'])
dailymeantemp_detrended = dailymeantemp_detrended.stack().stack().reorder_levels(['STATION','forecastDate_PST','variantID']).sort_index(level=['STATION','variantID','forecastDate_PST'])

# Rename variable names
dailymaxtemp_detrended = dailymaxtemp_detrended.rename(columns={'degF_detrended':'TEMP_MAX'})
dailymintemp_detrended = dailymintemp_detrended.rename(columns={'degF_detrended':'TEMP_MIN'})
dailymeantemp_detrended = dailymeantemp_detrended.rename(columns={'degF_detrended':'TEMP_MEAN'})

# Combine data into a single data frame
WS_daily = dailymaxtemp_detrended.merge(dailymintemp_detrended, left_index=True, right_index=True).merge(dailymeantemp_detrended, left_index=True, right_index=True)

# Calculate simple average of daily high and low temperatures, which is used in CDD/HDD calculations. Note that this is different from the daily mean temperature calculated above.
WS_daily['TEMP_AVG'] = (WS_daily['TEMP_MAX'] + WS_daily['TEMP_MIN']) / 2

# Calculate CDD and HDDs based on 65 degF threshold.
WS_daily['CDD65'] = (WS_daily['TEMP_AVG']-65).clip(lower=0)
WS_daily['HDD65'] = (65-WS_daily['TEMP_AVG']).clip(lower=0)

# Save the file in csv format
WS_daily.to_csv('outputs/daily/WS_daily_TEMP_detrended_all_' + date_stamp + '.csv')

for station_name, STATION in zip(station_list['station_name'].values, station_list['STATION'].values):
    print(station_name)
    df = WS_daily[WS_daily.index.get_level_values('STATION') == STATION]
    df.to_csv('outputs/daily/WS_daily_TEMP_detrended_' + STATION + '_' + date_stamp + '.csv')

# Delete unused data from memory 
del dailymaxtemp_detrended
del dailymintemp_detrended
del dailymeantemp_detrended

/var/folders/8t/v7mdtpss51qbx7bmcywwnw740000gn/T/ipykernel_38478/1987392525.py:19: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  dailymaxtemp_detrended = dailymaxtemp_detrended.stack().stack().reorder_levels(['STATION','forecastDate_PST','variantID']).sort_index(level=['STATION','variantID','forecastDate_PST'])
/var/folders/8t/v7mdtpss51qbx7bmcywwnw740000gn/T/ipykernel_38478/1987392525.py:19: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  dailymaxtemp_detrended = dailymaxtemp_detrended.stack().stack().reorder_levels(['STATION','forecastDate_PST','variantID']).sort_index(

Sacramento Executive Airport (KSAC)


### Step 2c. Annual and monthly CDD/HDDs by station
Calculate CDD65 and HDD65 metrics for all stations and variants. 

In [10]:
annual_CDD65 = WS_daily['CDD65'].unstack('variantID').unstack('STATION').resample('Y').sum()
annual_CDD65.index = annual_CDD65.index.year
annual_CDD65 = annual_CDD65.rename_axis(index='forecastYear')
annual_CDD65 = annual_CDD65.stack().reorder_levels(['STATION','forecastYear']).sort_index(level=['STATION', 'forecastYear'])
annual_CDD65.to_csv('outputs/annual_CDD65.csv')

annual_HDD65 = WS_daily['HDD65'].unstack('variantID').unstack('STATION').resample('Y').sum()
annual_HDD65.index = annual_HDD65.index.year
annual_HDD65 = annual_HDD65.rename_axis(index='forecastYear')
annual_HDD65 = annual_HDD65.stack().reorder_levels(['STATION','forecastYear']).sort_index(level=['STATION', 'forecastYear'])
annual_HDD65.to_csv('outputs/annual_HDD65.csv')

monthly_CDD65 = WS_daily['CDD65'].unstack('variantID').unstack('STATION').resample('M').sum()
monthly_CDD65 = monthly_CDD65.rename_axis(index='forecastMonth')
monthly_CDD65 = monthly_CDD65.stack().reorder_levels(['STATION','forecastMonth']).sort_index(level=['STATION', 'forecastMonth'])
monthly_CDD65.to_csv('outputs/monthly_CDD65.csv')

monthly_HDD65 = WS_daily['HDD65'].unstack('variantID').unstack('STATION').resample('M').sum()
monthly_HDD65 = monthly_HDD65.rename_axis(index='forecastMonth')
monthly_HDD65 = monthly_HDD65.stack().reorder_levels(['STATION','forecastMonth']).sort_index(level=['STATION', 'forecastMonth'])
monthly_HDD65.to_csv('outputs/monthly_HDD65.csv')

/var/folders/8t/v7mdtpss51qbx7bmcywwnw740000gn/T/ipykernel_38478/3234233426.py:1: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  annual_CDD65 = WS_daily['CDD65'].unstack('variantID').unstack('STATION').resample('Y').sum()
/var/folders/8t/v7mdtpss51qbx7bmcywwnw740000gn/T/ipykernel_38478/3234233426.py:4: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  annual_CDD65 = annual_CDD65.stack().reorder_levels(['STATION','forecastYear']).sort_index(level=['STATION', 'forecastYear'])
/var/folders/8t/v7mdtpss51qbx7bmcywwnw740000gn/T/ipykernel_38478/3234233426.py:7: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  annual_HDD65 = WS_daily['HDD65'].unstack('variantID').unstack('STATION').res

### Step 2d. Hourly temperatures by planning area
Aggregate hourly temperatures to planning area (PA) level by applying station weights.

In [11]:
WS_hourly_TEMP_all = None

for station_name, STATION in zip(station_list['station_name'].values, station_list['STATION'].values):
    print(station_name)
    WS_hourly_TEMP = pd.read_csv('outputs/hourly/temperature/WS_hourly_TEMP_detrended_' + STATION + '_' + date_stamp + '.csv')
    WS_hourly_TEMP['forecastDateTime_PST'] = pd.to_datetime(WS_hourly_TEMP['forecastDateTime_PST'])
    WS_hourly_TEMP['forecastDate_PST'] = pd.to_datetime(WS_hourly_TEMP['forecastDate_PST'])
    WS_hourly_TEMP['MONTH'] = WS_hourly_TEMP['forecastDate_PST'].dt.month
    WS_hourly_TEMP = WS_hourly_TEMP.set_index(['STATION','forecastDateTime_PST','forecastDate_PST','MONTH','Hour'])
    WS_hourly_TEMP = WS_hourly_TEMP.astype('float32')
    WS_hourly_TEMP_all = pd.concat([WS_hourly_TEMP_all, WS_hourly_TEMP])

WS_hourly_TEMP_all = WS_hourly_TEMP_all.reset_index()
PA_hourly = WS_hourly_TEMP_all.merge(station_weights, on=['STATION','MONTH'],how='outer')

for i in range(103, 154):
    PA_hourly['TEMP_variantID_' + str(i)] = PA_hourly['TEMP_variantID_' + str(i)] * PA_hourly['PA_WEIGHT']

PA_hourly.drop(['STATION','MONTH','PA_WEIGHT'], axis=1, inplace=True)
PA_hourly = PA_hourly.groupby(['PA_NAME','forecastDateTime_PST','forecastDate_PST','Hour']).sum().reset_index()
PA_hourly = PA_hourly.set_index(['PA_NAME','forecastDateTime_PST','forecastDate_PST','Hour'])
PA_hourly = PA_hourly.astype('float32')
PA_hourly

for pa_name in pd.unique(PA_hourly.index.get_level_values('PA_NAME')):
    df = PA_hourly[PA_hourly.index.get_level_values('PA_NAME') == pa_name]
    df.to_csv('outputs/hourly/temperature/PA_hourly_TEMP_detrended_' + pa_name + '_' + date_stamp + '.csv')

del WS_hourly_TEMP
del WS_hourly_TEMP_all
del df
#del PA_hourly

Sacramento Executive Airport (KSAC)


### Step 2e. Daily temperature stats by planning area
Two alternative approaches considered: (a) aggregate daily stats, and (b) aggregate hourly, then calculate daily stats.

In [12]:
# Calculate daily temperature stats (aggregate daily stats)
WS_daily_TEMP_all = None

for station_name, STATION in zip(station_list['station_name'].values, station_list['STATION'].values):
    print(station_name)
    WS_daily_TEMP = pd.read_csv('outputs/daily/WS_daily_TEMP_detrended_' + STATION + '_' + date_stamp + '.csv')
    WS_daily_TEMP['forecastDate_PST'] = pd.to_datetime(WS_daily_TEMP['forecastDate_PST'])
    WS_daily_TEMP['MONTH'] = WS_daily_TEMP['forecastDate_PST'].dt.month
    WS_daily_TEMP = WS_daily_TEMP.set_index(['STATION','forecastDate_PST','MONTH','variantID'])
    WS_daily_TEMP = WS_daily_TEMP.astype('float32')
    WS_daily_TEMP_all = pd.concat([WS_daily_TEMP_all, WS_daily_TEMP])

WS_daily_TEMP_all = WS_daily_TEMP_all.reset_index()
PA_daily = WS_daily_TEMP_all.merge(station_weights, on=['STATION','MONTH'],how='inner')
PA_daily['TEMP_MAX'] = PA_daily['TEMP_MAX'] * PA_daily['PA_WEIGHT']
PA_daily['TEMP_MIN'] = PA_daily['TEMP_MIN'] * PA_daily['PA_WEIGHT']
PA_daily['TEMP_MEAN'] = PA_daily['TEMP_MEAN'] * PA_daily['PA_WEIGHT']
PA_daily['TEMP_AVG'] = PA_daily['TEMP_AVG'] * PA_daily['PA_WEIGHT']
PA_daily['CDD65'] = PA_daily['CDD65'] * PA_daily['PA_WEIGHT']
PA_daily['HDD65'] = PA_daily['HDD65'] * PA_daily['PA_WEIGHT']

PA_daily.drop(['STATION','MONTH','PA_WEIGHT'], axis=1, inplace=True)
PA_daily = PA_daily.groupby(['PA_NAME','forecastDate_PST','variantID']).sum().reset_index()
PA_daily = PA_daily.sort_values(by=['PA_NAME','variantID','forecastDate_PST'])
PA_daily = PA_daily.set_index(['PA_NAME','forecastDate_PST','variantID'])
PA_daily = PA_daily.astype('float32')

for pa_name in pd.unique(PA_daily.index.get_level_values('PA_NAME')):
    df = PA_daily[PA_daily.index.get_level_values('PA_NAME') == pa_name]
    df.to_csv('outputs/daily/PA_daily_TEMP_detrended_' + pa_name + '_' + date_stamp + '.csv')

# Delete unused data from memory 
del WS_daily_TEMP
del WS_daily_TEMP_all
del PA_daily
del df

Sacramento Executive Airport (KSAC)


In [13]:
# Calculate daily temperature stats (aggregate hourly, then calculate daily stats)
PA_daily_TEMPmax = PA_hourly.reset_index().drop(['forecastDateTime_PST','Hour',], axis=1).groupby(['PA_NAME','forecastDate_PST']).max()
PA_daily_TEMPmax = PA_daily_TEMPmax.astype('float32')
PA_daily_TEMPmax = PA_daily_TEMPmax.reset_index().melt(id_vars=['PA_NAME','forecastDate_PST'], var_name='variantID', value_name='TEMP_MAX')
PA_daily_TEMPmax['variantID'] = PA_daily_TEMPmax['variantID'].str.extract(r'(\d+)').astype(int)
PA_daily_TEMPmax = PA_daily_TEMPmax.set_index(['PA_NAME','forecastDate_PST','variantID'])

PA_daily_TEMPmin = PA_hourly.reset_index().drop(['forecastDateTime_PST','Hour',], axis=1).groupby(['PA_NAME','forecastDate_PST']).min()
PA_daily_TEMPmin = PA_daily_TEMPmin.astype('float32')
PA_daily_TEMPmin = PA_daily_TEMPmin.reset_index().melt(id_vars=['PA_NAME','forecastDate_PST'], var_name='variantID', value_name='TEMP_MIN')
PA_daily_TEMPmin['variantID'] = PA_daily_TEMPmin['variantID'].str.extract(r'(\d+)').astype(int)
PA_daily_TEMPmin = PA_daily_TEMPmin.set_index(['PA_NAME','forecastDate_PST','variantID'])

PA_daily_TEMPmean = PA_hourly.reset_index().drop(['forecastDateTime_PST','Hour',], axis=1).groupby(['PA_NAME','forecastDate_PST']).mean()
PA_daily_TEMPmean = PA_daily_TEMPmean.astype('float32')
PA_daily_TEMPmean = PA_daily_TEMPmean.reset_index().melt(id_vars=['PA_NAME','forecastDate_PST'], var_name='variantID', value_name='TEMP_MEAN')
PA_daily_TEMPmean['variantID'] = PA_daily_TEMPmean['variantID'].str.extract(r'(\d+)').astype(int)
PA_daily_TEMPmean = PA_daily_TEMPmean.set_index(['PA_NAME','forecastDate_PST','variantID'])

# Combine data into a single data frame
PA_daily = PA_daily_TEMPmax.merge(PA_daily_TEMPmin, left_index=True, right_index=True).merge(PA_daily_TEMPmean, left_index=True, right_index=True)

# Calculate simple average of daily high and low temperatures, which is used in CDD/HDD calculations. Note that this is different from the daily mean temperature calculated above.
PA_daily['TEMP_AVG'] = (PA_daily['TEMP_MAX'] + PA_daily['TEMP_MIN']) / 2

# Calculate CDD and HDDs based on 65 degF threshold.
PA_daily['CDD65'] = (PA_daily['TEMP_AVG']-65).clip(lower=0)
PA_daily['HDD65'] = (65-PA_daily['TEMP_AVG']).clip(lower=0)

for pa_name in pd.unique(PA_daily.index.get_level_values('PA_NAME')):
    df = PA_daily[PA_daily.index.get_level_values('PA_NAME') == pa_name]
    df.to_csv('outputs/daily/PA_daily_TEMP_detrended_' + pa_name + '_' + date_stamp + '.csv')

# Delete unused data from memory 
del PA_daily_TEMPmax
del PA_daily_TEMPmin
del PA_daily_TEMPmean
del PA_daily
del df

# 3. Dew point inputs for demand forecasting

### Step 3a. Hourly dew point by station
For each variant, derive dew point using de-trended station-level temperatures and gridded relative humidity. Applied the same method used by AE team for consistency.  
https://github.com/cal-adapt/climakitae/blob/main/climakitae/tools/derived_variables.py

In [14]:
for station_name, STATION in zip(station_list['station_name'].values, station_list['STATION'].values):
    print(station_name)

    rh_ds = xr.open_dataset(f'inputs/{station_name}_rh_(new_core).nc')
    rh_ds = rh_ds.rename({'sim': 'simulation'})
    rh_ds = rh_ds.sel(simulation=list(simulation_rename.keys()))
    rh_ds = rh_ds.assign_coords(simulation=[simulation_rename[s] for s in rh_ds['simulation'].values])

    # Re-load hourly de-trended temperatures and reshape the data
    WS_hourly_TEMP = pd.read_csv('outputs/hourly/temperature/WS_hourly_TEMP_detrended_' + STATION + '_' + date_stamp + '.csv')
    WS_hourly_TEMP['forecastDateTime_PST'] = pd.to_datetime(WS_hourly_TEMP['forecastDateTime_PST'])
    WS_hourly_TEMP['forecastDate_PST'] = pd.to_datetime(WS_hourly_TEMP['forecastDate_PST'])
    WS_hourly_TEMP = WS_hourly_TEMP.set_index(['STATION','forecastDateTime_PST','forecastDate_PST','Hour'])
    WS_hourly_TEMP = WS_hourly_TEMP.astype('float32')
    WS_hourly_TEMP = WS_hourly_TEMP.reset_index().melt(id_vars=['STATION','forecastDateTime_PST','forecastDate_PST','Hour'], var_name='variantID', value_name='temperature')
    WS_hourly_TEMP['variantID'] = WS_hourly_TEMP['variantID'].str.extract(r'(\d+)').astype(int)

    # Add variant info
    WS_hourly_TEMP = WS_hourly_TEMP.merge(variants.reset_index(), on='variantID', how='inner')
    WS_hourly_TEMP['time'] = pd.to_datetime(dict({'year':WS_hourly_TEMP['forecastDateTime_PST'].dt.year + WS_hourly_TEMP['variantYear'],
                                                  'month':WS_hourly_TEMP['forecastDateTime_PST'].dt.month,
                                                  'day':WS_hourly_TEMP['forecastDateTime_PST'].dt.day,
                                                  'hour':WS_hourly_TEMP['forecastDateTime_PST'].dt.hour
                                                 }))

    # Relative humidity: this file was already narrowed to the station's nearest gridcell by
    # the clip processor in "Lumen fetch and localize WRF data (new_core).ipynb", so no
    # manual nearest-gridcell search is needed here (unlike the original notebook's argmin step)
    WS_hourly_humidity = rh_ds['relative_humidity_2m']
    del rh_ds
    WS_hourly_humidity = WS_hourly_humidity.expand_dims(scenario=['Historical + SSP 3-7.0 -- Business as Usual'])
    WS_hourly_humidity = WS_hourly_humidity.to_dataframe(name='relative_humidity').reset_index()
    WS_hourly_humidity = WS_hourly_humidity[['scenario','simulation','time','relative_humidity']]

    # Merge datasets for temperature and relative humidity
    WS_hourly = WS_hourly_TEMP.merge(WS_hourly_humidity, on=['scenario','simulation','time'], how='inner')
    del WS_hourly_TEMP
    del WS_hourly_humidity
    WS_hourly = WS_hourly[['STATION','forecastDateTime_PST','forecastDate_PST','Hour','variantID','temperature','relative_humidity']]
    WS_hourly = WS_hourly.sort_values(by=['STATION','variantID','forecastDateTime_PST'])
    WS_hourly = WS_hourly.set_index(['STATION','forecastDateTime_PST','forecastDate_PST','Hour','variantID'])

    # Calculate dew point based on temperature and relative humidity
    WS_hourly['temperature_K'] = (WS_hourly['temperature'] - 32) * (5/9) + 273.15
    WS_hourly['es'] = 0.611 * np.exp(5423 * ((1 / 273.15) - (1 / WS_hourly['temperature_K'])))
    WS_hourly['e_vap'] = (WS_hourly['es'] * WS_hourly['relative_humidity']) / 100.0
    WS_hourly['dew_point'] = ((1 / 273.15) - 0.0001844 * np.log(WS_hourly['e_vap']/ 0.611)) ** (-1)
    WS_hourly['dew_point'] = (WS_hourly['dew_point'] - 273.15) * (9/5) + 32
    WS_hourly = WS_hourly.astype('float32')

    # Reshape data and save as a CSV file
    WS_hourly = WS_hourly[['dew_point']].unstack('variantID').droplevel(level=0, axis=1)
    WS_hourly.columns = [f'DEWP_variantID_{col}' for col in WS_hourly.columns]
    WS_hourly.to_csv('outputs/hourly/dew_point/WS_hourly_DEWP_derived_' + STATION + '_' + date_stamp + '.csv')


Sacramento Executive Airport (KSAC)


### Step 3b. Hourly dew point by planning area
Aggregate hourly dew point to planning area (PA) level by applying station weights.

In [15]:
WS_hourly_DEWP_all = None

for station_name, STATION in zip(station_list['station_name'].values, station_list['STATION'].values):
    print(station_name)
    WS_hourly_DEWP = pd.read_csv('outputs/hourly/dew_point/WS_hourly_DEWP_derived_' + STATION + '_' + date_stamp + '.csv')
    WS_hourly_DEWP['forecastDateTime_PST'] = pd.to_datetime(WS_hourly_DEWP['forecastDateTime_PST'])
    WS_hourly_DEWP['forecastDate_PST'] = pd.to_datetime(WS_hourly_DEWP['forecastDate_PST'])
    WS_hourly_DEWP['MONTH'] = WS_hourly_DEWP['forecastDate_PST'].dt.month
    WS_hourly_DEWP = WS_hourly_DEWP.set_index(['STATION','forecastDateTime_PST','forecastDate_PST','MONTH','Hour'])
    WS_hourly_DEWP = WS_hourly_DEWP.astype('float32')
    WS_hourly_DEWP_all = pd.concat([WS_hourly_DEWP_all, WS_hourly_DEWP])

WS_hourly_DEWP_all = WS_hourly_DEWP_all.reset_index()
PA_hourly = WS_hourly_DEWP_all.merge(station_weights, on=['STATION','MONTH'],how='outer')

for i in range(103, 154):
    PA_hourly['DEWP_variantID_' + str(i)] = PA_hourly['DEWP_variantID_' + str(i)] * PA_hourly['PA_WEIGHT']

PA_hourly.drop(['STATION','MONTH','PA_WEIGHT'], axis=1, inplace=True)
PA_hourly = PA_hourly.groupby(['PA_NAME','forecastDateTime_PST','forecastDate_PST','Hour']).sum().reset_index()
PA_hourly = PA_hourly.set_index(['PA_NAME','forecastDateTime_PST','forecastDate_PST','Hour'])
PA_hourly = PA_hourly.astype('float32')
PA_hourly

for pa_name in pd.unique(PA_hourly.index.get_level_values('PA_NAME')):
    df = PA_hourly[PA_hourly.index.get_level_values('PA_NAME') == pa_name]
    df.to_csv('outputs/hourly/dew_point/PA_hourly_DEWP_derived_' + pa_name + '_' + date_stamp + '.csv')

del WS_hourly_DEWP
del WS_hourly_DEWP_all
del df
del PA_hourly

Sacramento Executive Airport (KSAC)


# Plot interactive summary charts
Summary charts illustrating de-trending of temperature projections. 

In [18]:
%config InlineBackend.figure_format = 'svg'

#create widgets
widget_station = pn.widgets.Select(options=list(quantiles.station.values))
widget_simulation = pn.widgets.Select(options=list(quantiles.simulation.values))
widget_year = pn.widgets.IntSlider(name='simulation year', value=2022, start=2022, end=2070)
widget_duration = pn.widgets.IntSlider(name='+/-X years for weather variants', value=15, start=10, end=25, step=5)
widget_trendrange = pn.widgets.IntRangeSlider(name= 'trend range', start=1981, end=2100, value=(1981, 2100), step=1)
widget_trendperiod = pn.widgets.RadioBoxGroup(name= 'trend period', options=['Long-Term', 'Rolling Window'])
widget_trendtype = pn.widgets.RadioBoxGroup(name= 'trend type', options=['Linear', '2nd Order'])
widget_bin = pn.widgets.RadioBoxGroup(name= 'quantile_bin', options=['Bin', 'No Bin'])

#set color palette and plot dimensions
color = hv.Palette('RdYlBu') #hv.Palette.colormaps.keys()
width = 600-45
height = 453+45
xlim = (1980, 2100)
ylim = (-0, 140)
bins = 150

quantiles_noBin = data_aggregated['degF'].groupby('time.year').quantile([0, .005, 0.03, .075, .125, .2, .3, .4, .5, .6, .7, .8, .875, .925, 0.97, .995, 1], dim='time')

#define the plotting function
def create_plots(station, simulation, year, duration, trendrange, trendperiod, trendtype, quantile_bin):
    
    if widget_bin.value == 'Bin':
        quantile_ds = quantiles.squeeze('scenario').rename({'bin':'quantile'})
        q_top = '99-100%'
        q_bottom = '00-01%'
        q_mid = '45-55%'
    elif widget_bin.value == 'No Bin':
        quantile_ds = quantiles_noBin.squeeze('scenario').to_dataset(name='degF')
        q_top = 1
        q_bottom = 0
        q_mid = 0.5
        
    quantile = quantile_ds.sel(year=slice(year-duration,year+duration))
    
    if widget_trendperiod.value == 'Long-Term':
        quantile_fortrend = quantile_ds.sel(year=slice(widget_trendrange.value[0],widget_trendrange.value[1]))
    elif widget_trendperiod.value == 'Rolling Window':
        quantile_fortrend = quantile_ds.sel(year=slice(year-duration,year+duration))
    
    if widget_trendtype.value == 'Linear':
        quantileFit = xr.polyval(quantile_fortrend['year'], quantile_fortrend.polyfit('year', 1).degF_polyfit_coefficients).to_dataset(name='degF')
        quantile_trend = quantile_fortrend.polyfit('year', 1).sel(degree=1).rename({'degF_polyfit_coefficients':'degF/yr'})
    elif widget_trendtype.value == '2nd Order':
        quantileFit = xr.polyval(quantile_fortrend['year'], quantile_fortrend.polyfit('year', 2).degF_polyfit_coefficients).to_dataset(name='degF')
        quantile_trend = quantile_fortrend.polyfit('year', 2).sel(degree=1).rename({'degF_polyfit_coefficients':'degF/yr'}) \
            + (2 * year * quantile_fortrend.polyfit('year', 2).sel(degree=2).rename({'degF_polyfit_coefficients':'degF/yr'}))
    
    quantileFit_y = quantileFit.sel(year=year)
    quantile_detrended = quantile - (quantileFit - quantileFit_y)
    
    plot1a = quantile_ds.sel(station=station, simulation=simulation).hvplot.line(
        title='Temperature Trends by Quantile \n(' + station + ', ' + simulation + ')',
        x='year', by='quantile',
        xlabel='', xlim=xlim, xticks=10,
        ylabel='degF', ylim=ylim, yticks=15,
        grid=True,
        width=width, height=height,  
        color=color,
        alpha=0.1,
        legend=False)

    plot1b = quantile.sel(station=station, simulation=simulation).hvplot.line(
        x='year', by='quantile',
        color=color,
        alpha=0.5,
        legend=True, fontsize={'legend': 9})

    plot1c = quantileFit.sel(station=station, simulation=simulation).hvplot.line(
        x='year', by='quantile',
        line_dash = ['dashed'],
        color=color,
        alpha=1.0,
        legend=False)

    plot1d = quantileFit_y.sel(station=station, simulation=simulation).hvplot.scatter(
        x='year', by='quantile',
        size=100,
        color=color,
        alpha=1.0, 
        legend=True)

    plot2a = quantile_detrended.sel(station=station, simulation=simulation).hvplot.line(
        title='De-Trended Temperatures by Quantile \n(' + station + ', ' + simulation + ')',
        x='year', by='quantile',
        xlabel='', xlim=xlim, xticks=10,
        ylabel='degF', ylim=ylim, yticks=15,
        grid=True,
        width=width, height=height,  
        color=color,
        alpha=1.0,
        legend=True, fontsize={'legend': 9})

    plot2b = quantile.sel(station=station, simulation=simulation).hvplot.line(
        x='year', by='quantile',
        color=color,
        alpha=0.3,
        legend=False)
        
    plot3a = quantile_detrended.sel(station=station, quantile=q_top).hvplot.hist(
        title='Distribution of Annual Min, Max & Median Temperatures \n(' + station + ', All GCMs)',
        xlabel='degF', xlim=ylim, xticks=15, bin_range=ylim, bins=bins,
        ylabel='# of variants',
        grid=True,
        width=width-100, height=height,  
        color=hv.Palette('Reds'),
        alpha=0.8,
        legend=False)
    
    plot3b = quantile_detrended.sel(station=station, quantile=q_bottom).hvplot.hist(
        xlabel='degF', xlim=ylim, xticks=15, bin_range=ylim, bins=bins,
        color=color,
        alpha=0.8,
        legend=False)
    
    plot3c = quantile_detrended.sel(station=station, quantile=q_mid).hvplot.hist(
        xlabel='degF', xlim=ylim, xticks=15, bin_range=ylim, bins=bins,
        color='Yellow',
        alpha=0.3,
        legend=False)
    
    plot4 = quantile_trend.sel(simulation=simulation).hvplot.scatter(
        title='Comparison of Temperature Trends across Stations \n(All stations, ' + simulation + ')',
        x='station', by='quantile',
        xlabel='', rot=90,
        ylabel='degF/yr', ylim=(-0.04,0.2), yticks=15,
        grid=True,
        width=width, height=height+150, 
        size=100,
        color=color,
        alpha=1.0,
        legend=False)
    
    plot4_alt = quantile_trend.sel(simulation=simulation).hvplot.heatmap(
        title='Comparison of Temperature Trends across Stations \n(All stations, ' + simulation + ')',
        x='station', y='quantile', yticks=10, C='degF/yr', clim=(-0.2, 0.2),
        xlabel='', rot=90,
        width=width, height=height+150, 
        cmap='RdBu_r',
        colorbar=True,
        legend=True)

    plot5 = quantileFit_y.sel(simulation=simulation).hvplot.scatter(
        title='Comparison of ' + str(year) + ' Fit across Stations \n(All stations, ' + simulation + ')',
        x='station', by='quantile',
        xlabel='', rot=90,
        ylabel='degF', ylim=ylim, yticks=15,
        grid=True,
        width=width, height=height+150, 
        size=100,
        color=color,
        alpha=1.0,
        legend=False)
    
    plot6 = quantileFit_y.sel(station=station).hvplot.scatter(
        title='Comparison of ' + str(year) + ' Fit across Simulations \n(' + station + ', All GCMs)',
        x='simulation', by='quantile',
        xlabel='', rot=90,
        ylabel='degF', ylim=ylim, yticks=15,
        grid=True,
        width=width-100, height=height+150,   
        size=100,
        color=color,
        alpha=1.0,
        legend=False)
    
    layout = hv.Layout((plot1a * plot1b * plot1c * plot1d) + (plot2a * plot2b) + (plot3a * plot3b * plot3c) + plot4 + plot5 + plot6).cols(3)
    
    return layout

app = pn.Column(pn.WidgetBox(pn.Spacer(height=10), pn.Row(pn.Column(widget_station, widget_simulation), pn.Column(widget_year, widget_duration, widget_trendrange), pn.Column(pn.Spacer(height=10), widget_trendperiod, pn.Spacer(height=5), widget_trendtype, pn.Spacer(height=5), widget_bin)), pn.Spacer(height=10)), pn.bind(create_plots, widget_station, widget_simulation, widget_year, widget_duration, widget_trendrange, widget_trendperiod, widget_trendtype, widget_bin))
app.servable()
app


BokehModel(combine_events=True, render_bundle={'docs_json': {'5f188891-2275-4eb7-b044-f3c577626c1a': {'version…